# 02. Очистка данных

Каждый этап - функция из `src/cleaning.py`. Блокнот вызывает их по одной и показывает результат; `src/pipeline.py` вызывает те же функции подряд.

Переменная `SAMPLE` переключает отладочный срез и полный набор.

In [1]:
import sys
from pathlib import Path

# блокнот запускается из notebooks/ - корень проекта нужен для импорта src
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import polars as pl
from IPython.display import Markdown, display

from src import cleaning, config, io_utils

pl.Config.set_tbl_cols(30)
pl.Config.set_tbl_width_chars(160)
pl.Config.set_fmt_str_lengths(60)

SAMPLE = False
src = config.variant(config.TYPED_PARQUET, SAMPLE)
df = pl.read_parquet(src)
cleaning.reset_stats()
print(df.shape)

(11540325, 19)


## rename_columns

Имена столбцов: нижний регистр, пробелы в подчёркивания.

In [2]:
before = df.height
df = cleaning.rename_columns(df)
print(f'строк было {before}, стало {df.height}')
print(df.columns)

строк было 11540325, стало 11540325
['geonameid', 'name', 'asciiname', 'alternatenames', 'latitude', 'longitude', 'feature_class', 'feature_code', 'country_code', 'cc2', 'admin1_code', 'admin2_code', 'admin3_code', 'admin4_code', 'population', 'elevation', 'dem', 'timezone', 'modification_date']


## cast_types

Явное приведение типов. Неразобранные значения становятся пропусками и считаются.

In [3]:
before = df.height
df = cleaning.cast_types(df)
print(f'строк было {before}, стало {df.height}')
display(df.schema)

строк было 11540325, стало 11540325


Schema([('geonameid', Int32),
        ('name', String),
        ('asciiname', String),
        ('alternatenames', String),
        ('latitude', Float32),
        ('longitude', Float32),
        ('feature_class', String),
        ('feature_code', String),
        ('country_code', String),
        ('cc2', String),
        ('admin1_code', String),
        ('admin2_code', String),
        ('admin3_code', String),
        ('admin4_code', String),
        ('population', Int64),
        ('elevation', Int16),
        ('dem', Int16),
        ('timezone', String),
        ('modification_date', Date)])

## replace_hidden_missing

Скрытые пропуски: `population = 0`, `dem = -9999`, пустые строки и служебные отметки.

In [4]:
before = df.height
df = cleaning.replace_hidden_missing(df)
print(f'строк было {before}, стало {df.height}')
display(cleaning.missing_profile(df).head(10))
display(cleaning.missing_by_group(df, 'population', 'feature_class'))

строк было 11540325, стало 11540325


column,nulls,share
str,u32,f64
"""cc2""",11222359,0.9724
"""population""",11042898,0.9569
"""admin4_code""",10932905,0.9474
"""admin3_code""",9510627,0.8241
"""elevation""",9189312,0.7963
"""alternatenames""",5954242,0.516
"""admin2_code""",5531864,0.4794
"""dem""",257561,0.0223
"""feature_code""",95368,0.0083


feature_class,rows,population_missing_share
str,u32,f64
"""P""",4577639,0.9197
"""S""",2340891,0.9989
"""H""",2186251,1.0
"""T""",1557088,0.9998
"""L""",400030,0.9992
"""A""",371320,0.6586
"""V""",45113,0.9998
"""R""",42400,0.9996
"""U""",14519,0.9999


## drop_full_duplicates

Полные дубликаты строк.

In [5]:
before = df.height
df = cleaning.drop_full_duplicates(df)
print(f'строк было {before}, стало {df.height}')

строк было 11540325, стало 11540325


## drop_key_duplicates

Дубликаты по ключу (имя, код, страна, координаты): остаётся последняя по дате запись.

In [6]:
before = df.height
df = cleaning.drop_key_duplicates(df)
print(f'строк было {before}, стало {df.height}')

строк было 11540325, стало 11538942


## normalize_categories

Категории: краевые пробелы, регистр кодов, упорядоченный список `cc2`.

In [7]:
before = df.height
df = cleaning.normalize_categories(df)
print(f'строк было {before}, стало {df.height}')
display(df['country_code'].value_counts(sort=True).head(10))

строк было 11538942, стало 11538942


country_code,count
str,u32
"""US""",2231558
"""CN""",735849
"""IN""",616599
"""NO""",600238
"""MX""",450403
"""ID""",383962
"""RU""",352182
"""CA""",313458
"""TH""",255104


## merge_rare_categories

Редкие коды объектов (меньше 100 записей) объединяются в группу `<класс>.OTHER` в отдельном столбце.

In [8]:
before = df.height
df = cleaning.merge_rare_categories(df)
print(f'строк было {before}, стало {df.height}')
display(df['feature_code_group'].value_counts(sort=True).tail(5))

строк было 11538942, стало 11538942


feature_code_group,count
str,u32
"""GOVL""",110
"""MNAU""",109
"""MGV""",109
"""CSWY""",109
"""P.OTHER""",22


## clean_text

Текст: разметка, повторные пробелы, пустые и повторяющиеся альтернативные названия.

In [9]:
before = df.height
df = cleaning.clean_text(df)
print(f'строк было {before}, стало {df.height}')
display(df.select('name', 'alternatenames').head(5))

строк было 11538942, стало 11538922


name,alternatenames
str,str
"""Zamīn Sūkhteh""","""Zamin Sukhteh;Zamīn Sūkhteh;zmyn swkhth;زمين سوخته"""
"""Rūdkhāneh-ye Zākalī""","""Ab-e Zalaki;Rudkhaneh-ye Zakali;Rūdkhāneh-ye Zākalī;rwdkhanh…"
"""Yekāhī""","""Yekahi;Yekāhī"""
"""Āb-e Yasī""",null
"""Tappeh-ye Seh Nūr""","""Tappeh-ye Seh Nur;Tappeh-ye Seh Nūr"""


## check_units

Единицы измерения: значения `elevation`, похожие на футы, помечаются флагом.

In [10]:
before = df.height
df = cleaning.check_units(df)
print(f'строк было {before}, стало {df.height}')
display(df.filter(pl.col('elevation_unit_suspect')).select('name', 'elevation', 'dem').head(5))

строк было 11538922, стало 11538922


name,elevation,dem
str,i16,i16
"""Jabal Bā Mashā’ib""",635,193
"""Jabal Sanām""",466,144
"""Jabal Qashkah""",1106,312
"""Jabal Kūl Dūbiryāw""",1335,404
"""Khashm Abraq al Ḩabārī""",847,253


## check_domain

Предметные ограничения: координаты, диапазоны высот и населения, даты.

In [11]:
before = df.height
df = cleaning.check_domain(df)
print(f'строк было {before}, стало {df.height}')

строк было 11538922, стало 11538922


## Выбросы: сравнение правил

Правила IQR и трёх сигм применяются к логарифму населения и к разности `elevation - dem`; результат сравнивается с предметным правилом.

In [12]:
display(cleaning.outlier_report(df))

variable,method,low,high,flagged
str,str,f64,f64,i64
"""log10(population)""","""IQR 1.5""",-0.7,5.87,3296
"""log10(population)""","""3 sigma""",-0.95,6.2,1710
"""log10(population)""","""предметное правило""",null,null,8
"""elevation - dem, м""","""IQR 1.5""",-14.0,10.0,274748
"""elevation - dem, м""","""3 sigma""",-175.34,174.02,18935
"""elevation - dem, м""","""предметное правило""",null,null,1972


На логарифме населения IQR и 3σ объявляют выбросами тысячи крупных городов: это правда, а не ошибка. Поэтому выбрано предметное правило: население больше 10 млн у гор, рек, зданий и подводных объектов и расхождение высот больше 1 км. Записи не удаляются, а помечаются флагами.

In [13]:
before = df.height
df = cleaning.flag_outliers(df)
print(f'строк было {before}, стало {df.height}')
display(df.filter(pl.col('population_outlier')).select('name', 'feature_class', 'country_code', 'population').sort('population', descending=True).head(8))

строк было 11538922, стало 11538922


name,feature_class,country_code,population
str,str,str,i64
"""Indian Subcontinent""","""T""","""IN""",1700000000
"""Balkan Peninsula""","""T""",null,60000000
"""Sumatra""","""T""","""ID""",50365538
"""Luzon""","""T""","""PH""",46228000
"""Godarzi It""","""S""","""IR""",44099591
"""Modaresi""","""S""","""IR""",22930913
"""Borneo""","""T""","""ID""",19000000
"""Kyushu""","""T""","""JP""",13231995


## Вспомогательные источники и соединение

In [14]:
health = cleaning.clean_health(io_utils.read_health())
countries = cleaning.build_countries(df, health)
display(countries.filter(pl.col('region').is_null()).head(5))

iso2,iso3,country_name,region,income_level
str,str,str,str,str
"""AI""","""AIA""","""Anguilla""",null,null
"""AN""","""ANT""","""Netherlands Antilles""",null,null
"""AQ""","""ATA""","""Antarctica""",null,null
"""AX""","""ALA""","""Åland Islands""",null,null
"""BL""","""BLM""","""Saint Barthélemy""",null,null


In [15]:
before = df.height
df = cleaning.join_countries(df, countries)
print(f'строк было {before}, стало {df.height}')
display(cleaning.STATS[-1])

строк было 11538922, стало 11538922


{'step': 'join_countries',
 'rows_in': 11538922,
 'rows_out': 11538922,
 'changed': 0,
 'removed': 0,
 'unmatched': 0,
 'no_country_code': 13742}

In [16]:
indicators, wdi = cleaning.clean_wdi(io_utils.read_wdi(), countries)
print(indicators.shape, wdi.shape)
display(cleaning.STATS[-1])

(345, 2) (2164493, 4)


{'step': 'clean_wdi',
 'rows_in': 2476042,
 'rows_out': 2164493,
 'changed': 0,
 'removed': 311549,
 'wide_rows': 89010,
 'unmatched_areas': 49}

## Итог: сводка «было - стало»

In [17]:
stats = cleaning.stats_frame()
display(stats.select('step', 'rows_in', 'rows_out', 'changed', 'removed'))
first = stats['rows_in'][0]
print(f'потеряно строк: {first - df.height} ({100 * (first - df.height) / first:.3f}%)')

потеряно строк: 1403 (0.012%)


step,rows_in,rows_out,changed,removed
str,i64,i64,i64,i64
"""rename_columns""",11540325,11540325,8,0
"""cast_types""",11540325,11540325,0,0
"""replace_hidden_missing""",11540325,11540325,11312856,0
"""drop_full_duplicates""",11540325,11540325,0,0
"""drop_key_duplicates""",11540325,11538942,0,1383
…,…,…,…,…
"""flag_outliers""",11538922,11538922,1980,0
"""clean_health""",5275,5275,2250,0
"""build_countries""",253,253,0,0


## Проверка согласованности со сценарием

Набор из блокнота сравнивается построчно с файлом, который записал `python -m src.pipeline`.

In [18]:
final = cleaning.finalize(df)
from_cli = pl.read_parquet(config.variant(config.CLEAN_PARQUET, SAMPLE))
same = final.equals(from_cli)
print("строк:", final.height, from_cli.height, "| совпадают построчно:", same)
assert same

строк: 11538922 11538922 | совпадают построчно: True
